# Fashion Image Preprocessing Pipeline
**Stages 3 & 4** — Deduplication, Reference Sorting, and No-Person Filtering

Run on Kaggle (T4 GPU) or Google Colab after Stages 01 & 02 have uploaded images to HuggingFace.

In [ ]:
!pip install -q open-clip-torch imagehash huggingface_hub pyarrow scikit-image opencv-python-headless groq

In [ ]:
import base64
import os
import shutil
import time
import logging
from io import BytesIO
from pathlib import Path

import cv2
import imagehash
import numpy as np
import open_clip
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from groq import Groq
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from skimage.metrics import structural_similarity as ssim
from tqdm.notebook import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

## Configuration

In [ ]:
HF_TOKEN          = "<INCLUDE HuggingFace Token Here"
HF_REPO_ID        = "prxkc/scraping-cv-pipeline"
GROQ_API_KEY      = "<Include Groq API Key Here>"
GROQ_VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
CLIP_MODEL        = "ViT-B-32"
CLIP_PRETRAINED   = "openai"
KEYWORDS          = ["panjabi", "burkha", "jubba", "thobe", "shalwar_kameez", "abaya"]

DEDUP_PHASH_THRESHOLD  = 8
DEDUP_HIST_THRESHOLD   = 0.98
DEDUP_SSIM_HIGH        = 0.92
DEDUP_SSIM_REVIEW_LOW  = 0.80
DEDUP_CLIP_THRESHOLD   = 0.97
SIMILARITY_THRESHOLD   = 0.70
NO_PERSON_THRESHOLD    = 0.75
BORDERLINE_LOW         = 0.50
SSIM_MAX_IMAGES        = 300

WORK_DIR   = Path("/kaggle/working") if os.path.exists("/kaggle/working") else Path("/content")
IMAGES_DIR = WORK_DIR / "images"
REF_DIR    = WORK_DIR / "reference_images"
SORTED_DIR = WORK_DIR / "sorted_output"
FINAL_DIR  = WORK_DIR / "final_output"
REVIEW_DIR = WORK_DIR / "review_queue"

for d in [IMAGES_DIR, REF_DIR, SORTED_DIR, FINAL_DIR, REVIEW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Download Reference Images

In [ ]:
print("Downloading reference images...")
for kw in KEYWORDS:
    dest = REF_DIR / f"{kw}_no_person.jpg"
    if not dest.exists():
        try:
            src = hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=f"reference_images/{kw}_no_person.jpg",
                repo_type="dataset",
                token=HF_TOKEN,
            )
            shutil.copy(src, dest)
            print(f"  {kw} ok")
        except Exception as e:
            print(f"  {kw} FAILED: {e}")

print("\nReference images ready:")
for f in sorted(REF_DIR.iterdir()):
    print(f"  {f.name}")

## CLIP Model

In [ ]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL, pretrained=CLIP_PRETRAINED
)
clip_model.eval().to(device)
print("CLIP loaded")


def encode_image(path):
    img = clip_preprocess(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = clip_model.encode_image(img)
    return F.normalize(feat, dim=-1).cpu()

## Stage 3A — Deduplication

In [ ]:
def phash_dedup(paths):
    hashes = {}
    for p in tqdm(paths, desc="pHash", leave=False):
        try:
            hashes[p] = imagehash.phash(Image.open(p))
        except Exception:
            pass
    path_list = list(hashes)
    to_remove = set()
    for i, p1 in enumerate(path_list):
        if p1 in to_remove:
            continue
        for p2 in path_list[i + 1:]:
            if p2 not in to_remove and hashes[p1] - hashes[p2] <= DEDUP_PHASH_THRESHOLD:
                to_remove.add(p2)
    kept = [p for p in path_list if p not in to_remove]
    print(f"  pHash: {len(kept)} kept, {len(to_remove)} removed")
    return kept


def histogram_dedup(paths):
    def hist(p):
        img = cv2.imread(str(p))
        if img is None:
            return np.zeros(768)
        h = [cv2.calcHist([img], [c], None, [256], [0, 256]) for c in range(3)]
        return np.concatenate([cv2.normalize(x, x).flatten() for x in h])

    histograms = {p: hist(p) for p in tqdm(paths, desc="Histogram", leave=False)}
    path_list = list(histograms)
    to_remove = set()
    for i, p1 in enumerate(path_list):
        if p1 in to_remove:
            continue
        h1 = histograms[p1].reshape(-1, 1).astype(np.float32)
        for p2 in path_list[i + 1:]:
            if p2 not in to_remove:
                h2 = histograms[p2].reshape(-1, 1).astype(np.float32)
                if cv2.compareHist(h1, h2, cv2.HISTCMP_CORREL) > DEDUP_HIST_THRESHOLD:
                    to_remove.add(p2)
    kept = [p for p in path_list if p not in to_remove]
    print(f"  Histogram: {len(kept)} kept, {len(to_remove)} removed")
    return kept


def ssim_dedup(paths, review_dir):
    def load_gray(p):
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        return cv2.resize(img, (256, 256)) if img is not None else None

    grays = {p: load_gray(p) for p in tqdm(paths, desc="SSIM", leave=False)}
    path_list = [p for p, g in grays.items() if g is not None]
    to_remove, flagged = set(), 0
    review_dir.mkdir(parents=True, exist_ok=True)
    for i, p1 in enumerate(path_list):
        if p1 in to_remove:
            continue
        for p2 in path_list[i + 1:]:
            if p2 in to_remove:
                continue
            score, _ = ssim(grays[p1], grays[p2], full=True)
            if score > DEDUP_SSIM_HIGH:
                to_remove.add(p2)
            elif score > DEDUP_SSIM_REVIEW_LOW:
                shutil.copy2(p2, review_dir / f"review_{p1.stem}_vs_{p2.stem}_{score:.3f}{p2.suffix}")
                flagged += 1
    kept = [p for p in path_list if p not in to_remove]
    print(f"  SSIM: {len(kept)} kept, {len(to_remove)} removed, {flagged} flagged")
    return kept


def clip_dedup(paths):
    embeddings, valid = [], []
    for p in tqdm(paths, desc="CLIP dedup", leave=False):
        try:
            embeddings.append(encode_image(p))
            valid.append(p)
        except Exception:
            pass
    if not embeddings:
        return paths
    mat = torch.cat(embeddings)
    sim = mat @ mat.T
    n = len(valid)
    to_remove = set()
    for i in range(n):
        if i in to_remove:
            continue
        for j in range(i + 1, n):
            if j not in to_remove and sim[i, j].item() > DEDUP_CLIP_THRESHOLD:
                to_remove.add(j)
    kept = [valid[i] for i in range(n) if i not in to_remove]
    print(f"  CLIP: {len(kept)} kept, {len(to_remove)} removed")
    return kept


def deduplicate(paths, review_dir):
    r = phash_dedup(paths)
    r = histogram_dedup(r)
    if len(r) <= SSIM_MAX_IMAGES:
        r = ssim_dedup(r, review_dir)
    else:
        print(f"  SSIM: skipped ({len(r)} images)")
    r = clip_dedup(r)
    print(f"  Total: {len(r)}/{len(paths)} retained")
    return r

## Stage 3B — Reference Sorting & HTML Report

In [ ]:
def reference_sort(paths, ref_path, threshold):
    ref_embed = encode_image(ref_path)
    results = []
    for p in tqdm(paths, desc="Similarity scoring", leave=False):
        try:
            score = F.cosine_similarity(ref_embed, encode_image(p)).item()
            if score >= threshold:
                results.append((p, score))
        except Exception:
            pass
    results.sort(key=lambda x: -x[1])
    return results


def save_sorted(results, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    for rank, (src, score) in enumerate(results, 1):
        shutil.copy2(src, out_dir / f"rank{rank:04d}_sim{score:.3f}{src.suffix.lower()}")


def build_report(results, ref_path, keyword, out_dir):
    cards = ""
    for rank, (p, score) in enumerate(results, 1):
        img_name = f"rank{rank:04d}_sim{score:.3f}{p.suffix.lower()}"
        pct = int(score * 100)
        cards += f'<div class="card"><img src="{img_name}" loading="lazy"><div class="meta">#{rank} {score:.3f}<progress value="{pct}" max="100"></progress></div></div>'
    ref_name = ref_path.name
    html = (
        f"<!DOCTYPE html><html><head><meta charset=UTF-8>"
        f"<title>{keyword}</title><style>"
        "body{background:#111;color:#eee;font-family:sans-serif}"
        "h1{text-align:center}"
        ".ref{display:flex;justify-content:center;margin:1rem}"
        ".ref img{max-height:280px;border:3px solid #4af;border-radius:8px}"
        ".grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(180px,1fr));gap:1rem}"
        ".card{background:#222;border-radius:8px;overflow:hidden}"
        ".card img{width:100%;height:180px;object-fit:cover}"
        ".meta{padding:.4rem;font-size:.8rem}"
        "progress{width:100%;height:5px}"
        f"</style></head><body>"
        f"<h1>{keyword} — {len(results)} matches</h1>"
        f'<div class=ref><img src="{ref_name}" alt=reference></div>'
        f"<div class=grid>{cards}</div></body></html>"
    )
    (out_dir / "report.html").write_text(html, encoding="utf-8")

## Stage 4 — No-Person Filter (CLIP + Groq LLM)

In [ ]:
groq_client = Groq(api_key=GROQ_API_KEY)
_last_llm_call = 0.0
_LLM_MIN_INTERVAL = 4.0


def image_to_b64(path):
    buf = BytesIO()
    Image.open(path).convert("RGB").save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode()


def llm_has_person(path):
    global _last_llm_call
    wait = _LLM_MIN_INTERVAL - (time.time() - _last_llm_call)
    if wait > 0:
        time.sleep(wait)
    try:
        resp = groq_client.chat.completions.create(
            model=GROQ_VISION_MODEL,
            messages=[{"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_to_b64(path)}"}},
                {"type": "text", "text": "Does this image contain a visible person? Respond only YES or NO."},
            ]}],
            max_tokens=5,
            temperature=0.0,
        )
        return resp.choices[0].message.content.strip().upper().startswith("YES")
    except Exception as e:
        log.warning("LLM vision error %s: %s", path.name, e)
        return False
    finally:
        _last_llm_call = time.time()


def person_filter(sorted_dir, ref_path, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    ref_embed = encode_image(ref_path)
    exts = {".jpg", ".jpeg", ".png", ".webp"}
    candidates = sorted(p for p in sorted_dir.iterdir() if p.suffix.lower() in exts)
    final, kept, excl, reviewed = [], 0, 0, 0
    for img in tqdm(candidates, desc=f"Filtering {sorted_dir.name}"):
        try:
            score = F.cosine_similarity(ref_embed, encode_image(img)).item()
        except Exception:
            excl += 1
            continue
        if score >= NO_PERSON_THRESHOLD:
            final.append((img, score))
            kept += 1
        elif score >= BORDERLINE_LOW:
            reviewed += 1
            if not llm_has_person(img):
                final.append((img, score))
                kept += 1
            else:
                excl += 1
        else:
            excl += 1
    final.sort(key=lambda x: -x[1])
    for rank, (src, score) in enumerate(final, 1):
        shutil.copy2(src, out_dir / f"rank{rank:04d}_sim{score:.3f}{src.suffix.lower()}")
    print(f"  kept={kept}, excluded={excl}, LLM-reviewed={reviewed} -> {len(final)} images")
    return final

## Run Full Pipeline

In [ ]:
api = HfApi(token=HF_TOKEN)

for keyword in KEYWORDS:
    print(f"\n{chr(61)*60}\nProcessing: {keyword}\n{chr(61)*60}")
    kw_dir = IMAGES_DIR / keyword
    ref_path = REF_DIR / f"{keyword}_no_person.jpg"

    if not kw_dir.exists():
        kw_dir.mkdir(parents=True, exist_ok=True)
        try:
            all_files = list(api.list_repo_files(HF_REPO_ID, repo_type="dataset", token=HF_TOKEN))
            parquet_files = [f for f in all_files if f.startswith(f"data/{keyword}/") and f.endswith(".parquet")]
            print(f"  Downloading {len(parquet_files)} shard(s)...")
            for pq_file in tqdm(parquet_files, desc="Shards"):
                local = hf_hub_download(repo_id=HF_REPO_ID, filename=pq_file, repo_type="dataset", token=HF_TOKEN)
                table = pq.read_table(local)
                for i in range(table.num_rows):
                    fname = table.column("filename")[i].as_py()
                    img_bytes = table.column("image_bytes")[i].as_py()
                    dest = kw_dir / fname
                    if not dest.exists():
                        dest.write_bytes(img_bytes)
        except Exception as e:
            print(f"  HF load failed: {e} — skipping")
            continue

    exts = {".jpg", ".jpeg", ".png", ".webp"}
    paths = sorted(p for p in kw_dir.iterdir() if p.suffix.lower() in exts)
    if not paths:
        print("  No images found — skipping")
        continue
    print(f"  {len(paths)} images loaded")

    deduped = deduplicate(paths, REVIEW_DIR / keyword)

    if not ref_path.exists():
        print(f"  Reference image not found — skipping 3B & 4")
        continue

    sorted_dir = SORTED_DIR / keyword
    results = reference_sort(deduped, ref_path, SIMILARITY_THRESHOLD)
    print(f"  Matching images for '{keyword}': {len(results)}")
    save_sorted(results, sorted_dir)
    build_report(results, ref_path, keyword, sorted_dir)

    person_filter(sorted_dir, ref_path, FINAL_DIR / keyword)

print("\nPipeline complete:")
for kw in KEYWORDS:
    n = len(list((FINAL_DIR / kw).glob("rank*"))) if (FINAL_DIR / kw).exists() else 0
    print(f"  {kw}: {n} images")

## Display Results Sample

In [ ]:
import matplotlib.pyplot as plt

for keyword in KEYWORDS:
    out = FINAL_DIR / keyword
    if not out.exists():
        continue
    imgs = sorted(out.glob("rank*"))[:6]
    if not imgs:
        continue
    fig, axes = plt.subplots(1, len(imgs), figsize=(3 * len(imgs), 3))
    if len(imgs) == 1:
        axes = [axes]
    fig.suptitle(keyword, fontsize=14)
    for ax, p in zip(axes, imgs):
        ax.imshow(Image.open(p).convert("RGB"))
        ax.set_title(p.stem[-10:], fontsize=7)
        ax.axis("off")
    plt.tight_layout()
    plt.show()